In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler


In [3]:

# Download data
stock = "GOOG"
start = "2012-01-01"
end = "2022-12-31"

data = yf.download(stock, start, end)


[*********************100%***********************]  1 of 1 completed


In [5]:
data = data[['Close']]
split_index = int(len(data) * 0.8)

data_train = data[:split_index]
data_test = data[split_index:]

scaler = MinMaxScaler(feature_range=(0,1))
data_train_scaled = scaler.fit_transform(data_train)

x = []
y = []

for i in range(100, data_train_scaled.shape[0]):
    x.append(data_train_scaled[i-100:i])
    y.append(data_train_scaled[i,0])

x = np.array(x)
y = np.array(y)

x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1,1)

# LSTM Model
class StockLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm1 = nn.LSTM(1,50,batch_first=True)
        self.dropout1 = nn.Dropout(0.2)

        self.lstm2 = nn.LSTM(50,60,batch_first=True)
        self.dropout2 = nn.Dropout(0.3)

        self.lstm3 = nn.LSTM(60,80,batch_first=True)
        self.dropout3 = nn.Dropout(0.4)

        self.lstm4 = nn.LSTM(80,120,batch_first=True)
        self.dropout4 = nn.Dropout(0.5)

        self.fc = nn.Linear(120,1)

    def forward(self,x):

        x,_ = self.lstm1(x)
        x = self.dropout1(x)

        x,_ = self.lstm2(x)
        x = self.dropout2(x)

        x,_ = self.lstm3(x)
        x = self.dropout3(x)

        x,_ = self.lstm4(x)
        x = self.dropout4(x)

        x = x[:,-1,:]
        x = self.fc(x)

        return x


model = StockLSTM()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
batch_size = 32

for epoch in range(epochs):

    for i in range(0,len(x),batch_size):

        batch_x = x[i:i+batch_size]
        batch_y = y[i:i+batch_size]

        optimizer.zero_grad()

        outputs = model(batch_x)

        loss = criterion(outputs,batch_y)

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs} Loss: {loss.item()}")

torch.save(model.state_dict(),"stock_model.pth")

print("Model trained and saved as stock_model.pth")

Epoch 1/20 Loss: 0.05665256083011627
Epoch 2/20 Loss: 0.007701103575527668
Epoch 3/20 Loss: 0.03011249750852585
Epoch 4/20 Loss: 0.2497643679380417
Epoch 5/20 Loss: 0.16348141431808472
Epoch 6/20 Loss: 0.21962958574295044
Epoch 7/20 Loss: 0.22064587473869324
Epoch 8/20 Loss: 0.07590696215629578
Epoch 9/20 Loss: 0.1484619677066803
Epoch 10/20 Loss: 0.19807404279708862
Epoch 11/20 Loss: 0.18977153301239014
Epoch 12/20 Loss: 0.24718797206878662
Epoch 13/20 Loss: 0.22623097896575928
Epoch 14/20 Loss: 0.20462557673454285
Epoch 15/20 Loss: 0.21210208535194397
Epoch 16/20 Loss: 0.2449338436126709
Epoch 17/20 Loss: 0.2417147010564804
Epoch 18/20 Loss: 0.24095377326011658
Epoch 19/20 Loss: 0.20540949702262878
Epoch 20/20 Loss: 0.21593084931373596
Model trained and saved as stock_model.pth
